# Market-Signal-Enhanced Delta Hedging on Historical Equity Paths

This notebook is the primary empirical report. It asks whether public point-in-time market signals improve replication of standardized European-call payoffs relative to fixed-volatility Black-Scholes hedging.

> **Scope:** SPY paths and hedge signals are observed market data. The option claims are controlled, model-defined European payoffs—not historical exchange quotes.

## Design

The experiment uses chronological training, validation, and held-out test periods. Validation selects one of three predeclared composite-risk configurations. The test grid crosses 10/21/42 trading-day maturities with strike/spot ratios of 0.95/1.00/1.05 and starts every five trading days. All strategies receive the same FV-BS premium within an episode and pay 5 bps proportional transaction costs.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, ".")
from examples.run_real_market_pipeline import run_real_market_pipeline, save_outputs
result = run_real_market_pipeline()
save_outputs(result)
result.split_dates, result.selected_configuration

({'train': '2016-04-05 to 2020-08-14',
  'validation': '2020-08-17 to 2022-05-13',
  'test': '2022-05-16 to 2024-12-31'},
 RiskConfiguration(name='moderate', elevated_quantile=0.7, stress_quantile=0.9, elevated_multiplier=1.15, stress_multiplier=1.4))

## Validation selection

In [2]:
result.tuning_results.round(4)

,name,elevated_quantile,stress_quantile,elevated_multiplier,stress_multiplier,validation_episodes,validation_mae,validation_rmse
1,moderate,0.70,0.90,1.15,1.40,42,1.8893,2.5794
2,responsive,0.60,0.85,1.25,1.60,42,1.9140,2.5534
0,conservative,0.75,0.92,1.10,1.25,42,2.0016,2.6857


## Held-out results

In [3]:
result.hedging_metrics.round(4)

,n_episodes,mean_error,mae,rmse,error_std,max_absolute_error,mean_transaction_cost,mean_turnover_notional,absolute_error_q90,loss_var_90,loss_es_90,absolute_error_q95,loss_var_95,loss_es_95
strategy,,,,,,,,,,,,,,
FV-BS,1149,-0.2265,1.5523,2.5422,2.5332,13.7308,0.4631,926.1750,3.8674,2.6424,4.9816,5.4631,3.8854,6.7462
MSA-Delta,1149,-0.2417,1.6906,2.5368,2.5263,10.1594,0.4662,932.4539,4.2435,3.0294,5.0937,5.7269,4.3320,6.5818
Rolling-BS,1149,-0.2492,1.6073,2.4385,2.4268,9.9717,0.4690,937.9716,4.1343,2.9127,5.0327,5.4628,4.3268,6.4496
VIX-BS,1149,-0.1950,1.6956,2.3993,2.3924,12.7174,0.4580,915.9682,4.0538,2.7725,4.4574,5.2032,4.0652,5.5170


In [4]:
result.bootstrap_intervals.round(4)

,mean_mae_improvement,ci_2_5,ci_97_5,probability_improvement
strategy,,,,
MSA-Delta,-0.1379,-0.2543,-0.0390,0.0020
Rolling-BS,-0.0566,-0.1189,0.0064,0.0400
VIX-BS,-0.1481,-0.2549,-0.0330,0.0065


## Robustness

The next table separates performance by maturity and moneyness; it prevents the pooled average from hiding contract-specific behavior.

In [5]:
from IPython.display import display
display(result.robustness_metrics[["maturity_days", "moneyness", "strategy", "n_episodes", "mae", "rmse", "loss_es_90"]].round(4))
display(result.cost_sensitivity[["transaction_cost_rate", "strategy", "mae", "rmse"]].round(4))

,maturity_days,moneyness,strategy,n_episodes,mae,rmse,loss_es_90
0,10,0.95,FV-BS,131,0.6427,0.9110,2.0903
1,10,0.95,MSA-Delta,131,0.7610,1.0421,2.4885
2,10,0.95,Rolling-BS,131,0.6728,0.9868,2.3388
3,10,0.95,VIX-BS,131,0.8253,1.0764,2.4173
4,10,1.00,FV-BS,131,1.5793,2.3060,5.0912
5,10,1.00,MSA-Delta,131,1.5736,2.2569,4.8597
6,10,1.00,Rolling-BS,131,1.5714,2.2778,5.0347
7,10,1.00,VIX-BS,131,1.5999,2.1553,4.5488
8,10,1.05,FV-BS,131,0.5277,1.0638,1.0492
9,10,1.05,MSA-Delta,131,0.6351,1.2512,1.4125


,transaction_cost_rate,strategy,mae,rmse
0,0.0000,FV-BS,1.4565,2.4824
1,0.0000,MSA-Delta,1.6201,2.5089
2,0.0000,Rolling-BS,1.5358,2.3943
3,0.0000,VIX-BS,1.6111,2.3939
4,0.0005,FV-BS,1.5523,2.5422
5,0.0005,MSA-Delta,1.6906,2.5368
6,0.0005,Rolling-BS,1.6073,2.4385
7,0.0005,VIX-BS,1.6956,2.3993
8,0.0010,FV-BS,1.7454,2.7045
9,0.0010,MSA-Delta,1.8530,2.6695


## Conclusion

The larger held-out experiment does **not** show that additional signal complexity uniformly improves hedging. FV-BS has the lowest mean absolute replication error. VIX-BS has the lowest RMSE and 95% loss expected shortfall, consistent with option-market information helping in large-error episodes. The validation-selected MSA-Delta rule does not beat the simpler benchmarks on MAE.

The appropriate conclusion is therefore conditional: forward-looking market volatility appears useful for tail-risk control, but the hand-designed composite state rule adds no robust average-error advantage in this sample.

### Limitations

The study does not observe historical option bids, asks, early exercise, contract-specific implied volatility, or volatility surfaces. VIX is an index-level proxy, and moving-block bootstrap inference for overlapping episodes remains approximate.